# Inspección de Features SAE desde Tablero

Este notebook permite:
1. Tomar un tablero de Othello
2. Obtener activaciones del modelo GPT (layer 6)
3. Pasar por el SAE encoder
4. Ver qué features se activan más

In [1]:
import numpy as np
import torch
import sys
from pathlib import Path

# Añadir raíz del proyecto al path
project_root = Path.cwd().parent.parent
sys.path.insert(0, str(project_root))

print(f"Proyecto: {project_root}")

Proyecto: c:\Users\Esposa\Documents\Repos\othello_world


## 1. Cargar Modelos

In [2]:
from mingpt.model import GPT, GPTConfig
from sae.models.sae import SparseAutoencoder

# Configuración
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Device: {device}")

# Configuración del modelo GPT (Othello-GPT)
mconf = GPTConfig(
    vocab_size=61,  # Othello: 60 movimientos + 1 padding
    block_size=59,  # Longitud máxima de secuencia
    n_layer=8,
    n_head=8,
    n_embd=512
)

# Cargar modelo GPT
gpt_checkpoint = project_root / "ckpts" / "gpt_championship.ckpt"
print(f"\nCargando GPT desde: {gpt_checkpoint}")

gpt_model = GPT(mconf)
gpt_model.load_state_dict(torch.load(gpt_checkpoint, map_location=device))
gpt_model = gpt_model.to(device)
gpt_model.eval()

print(f"✓ GPT cargado: {mconf.n_layer} layers, {mconf.n_embd} dim")

Device: cuda

Cargando GPT desde: c:\Users\Esposa\Documents\Repos\othello_world\ckpts\gpt_championship.ckpt
✓ GPT cargado: 8 layers, 512 dim


In [3]:
# Cargar SAE
# TODO: Ajustar la ruta según dónde esté tu modelo SAE entrenado
sae_path = project_root / "sae" / "experiments" / "layer_06" / "sae_best.pt"

if not sae_path.exists():
    print(f"  No se encontró SAE en: {sae_path}")
    print("Por favor ajusta la ruta en la celda anterior")
else:
    print(f"\nCargando SAE desde: {sae_path}")
    sae_checkpoint = torch.load(sae_path, map_location=device)
    
    # Crear instancia del SAE
    input_dim = gpt_model.config.n_embd  # 512 para Othello-GPT
    hidden_dim = sae_checkpoint.get('hidden_dim', 8192)  # Ajustar según tu SAE
    
    sae = SparseAutoencoder(input_dim=input_dim, hidden_dim=hidden_dim)
    sae.load_state_dict(sae_checkpoint['model_state_dict'])
    sae = sae.to(device)
    sae.eval()
    
    print(f"✓ SAE cargado: {input_dim} → {hidden_dim} features")

  No se encontró SAE en: c:\Users\Esposa\Documents\Repos\othello_world\sae\experiments\layer_06\sae_best.pt
Por favor ajusta la ruta en la celda anterior


## 2. Función Principal: Tablero → Top Features

In [4]:
def tablero_to_sequence(tablero, color_jugador=1):
    """
    Convierte un tablero 8x8 en una secuencia tokenizada para el modelo.
    
    Args:
        tablero: array 8x8 con valores {-1, 0, 1}
        color_jugador: 1 (negro) o -1 (blanco)
    
    Returns:
        tokens: tensor de shape (1, seq_len) para el modelo
    """
    # Convertir tablero a estado del juego
    # El modelo espera secuencia de movimientos, pero podemos usar
    # la representación del tablero directamente si solo necesitamos activaciones
    
    # Por simplicidad, mapeamos el tablero a una secuencia flat
    # Mapeo: -1 (blanco) → 0, 0 (vacío) → 1, 1 (negro) → 2
    flat = tablero.flatten()
    tokens = flat + 1  # Shift para tener valores {0, 1, 2}
    
    # Convertir a tensor y agregar batch dimension
    tokens = torch.tensor(tokens, dtype=torch.long, device=device).unsqueeze(0)
    
    return tokens


def get_layer_activations(model, tokens, layer_idx=5):
    """
    Extrae activaciones de una capa específica del modelo GPT.
    
    Args:
        model: Modelo GPT
        tokens: Input tokens (B, T)
        layer_idx: Índice de la capa (0-indexed, layer 6 = idx 5)
    
    Returns:
        activations: tensor de shape (B, T, n_embd)
    """
    activations = None
    
    def hook_fn(module, input, output):
        nonlocal activations
        activations = output[0].detach()  # output es tuple (x, present)
    
    # Registrar hook en la capa deseada
    hook = model.transformer.h[layer_idx].register_forward_hook(hook_fn)
    
    # Forward pass
    with torch.no_grad():
        _ = model(tokens)
    
    hook.remove()
    return activations


def inspect_top_features(tablero, sae, top_k=3, layer_idx=5, verbose=True):
    """
    Función principal: Tablero → Top Features SAE
    
    Args:
        tablero: array 8x8 numpy con valores {-1, 0, 1}
        sae: Modelo SparseAutoencoder
        top_k: Número de features a mostrar
        layer_idx: Capa del GPT a extraer (default: 5 = layer 6)
        verbose: Si mostrar output detallado
    
    Returns:
        top_features: Lista de tuplas (feature_idx, activation_value)
    """
    # 1. Convertir tablero a tokens
    tokens = tablero_to_sequence(tablero)
    
    # 2. Extraer activaciones del GPT (layer 6)
    activations = get_layer_activations(gpt_model, tokens, layer_idx=layer_idx)
    # activations shape: (1, seq_len, n_embd)
    
    # 3. Pasar por SAE encoder
    with torch.no_grad():
        # Reshape para pasar por SAE: (batch * seq_len, n_embd)
        batch_size, seq_len, n_embd = activations.shape
        flat_acts = activations.reshape(-1, n_embd)
        
        # Encode
        sae_features = sae.encode(flat_acts)  # (batch * seq_len, hidden_dim)
        
        # Promediar sobre la secuencia
        sae_features = sae_features.reshape(batch_size, seq_len, -1)
        sae_features_avg = sae_features.mean(dim=1)  # (batch, hidden_dim)
    
    # 4. Obtener top-k features
    feature_values = sae_features_avg[0].cpu().numpy()  # (hidden_dim,)
    top_indices = np.argsort(feature_values)[::-1][:top_k]
    top_values = feature_values[top_indices]
    
    # Crear lista de resultados
    top_features = [(int(idx), float(val)) for idx, val in zip(top_indices, top_values)]
    
    # 5. Mostrar resultados
    if verbose:
        print(f"\n Top {top_k} Features SAE más activadas:")
        print("=" * 40)
        for idx, val in top_features:
            print(f"Feature #{idx:4d}:  {val:6.2f}")
        print("=" * 40)
    
    return top_features

## 3. Utilidades: Visualización de Tablero

In [5]:
def imprimir_tablero(tablero):
    """Visualiza el tablero de Othello."""
    print("  A B C D E F G H")
    for i, fila in enumerate(tablero):
        print(f"{i+1}", end=" ")
        for valor in fila:
            if valor == 0:
                print(".", end=" ")
            elif valor == 1:
                print("●", end=" ")  # Negra
            else:
                print("○", end=" ")  # Blanca
        print(f"{i+1}")
    print("  A B C D E F G H")

## 4. Ejemplos de Uso

### Ejemplo 1: Tablero Inicial

In [6]:
# Crear tablero en estado inicial
tablero_inicial = np.array([
    [0, 0, 0, 0, 0, 0, 0, 0],
    [0, 0, 0, 0, 0, 0, 0, 0],
    [0, 0, 0, 0, 0, 0, 0, 0],
    [0, 0, 0, -1, 1, 0, 0, 0],  # D4=blanca(-1), E4=negra(1)
    [0, 0, 0, 1, -1, 0, 0, 0],  # D5=negra(1), E5=blanca(-1)
    [0, 0, 0, 0, 0, 0, 0, 0],
    [0, 0, 0, 0, 0, 0, 0, 0],
    [0, 0, 0, 0, 0, 0, 0, 0]
])

print("TABLERO INICIAL:")
imprimir_tablero(tablero_inicial)

# Inspeccionar features
top_features = inspect_top_features(tablero_inicial, sae, top_k=5)

TABLERO INICIAL:
  A B C D E F G H
1 . . . . . . . . 1
2 . . . . . . . . 2
3 . . . . . . . . 3
4 . . . ○ ● . . . 4
5 . . . ● ○ . . . 5
6 . . . . . . . . 6
7 . . . . . . . . 7
8 . . . . . . . . 8
  A B C D E F G H


NameError: name 'sae' is not defined

### Ejemplo 2: Tablero Personalizado

In [ ]:
# Crear tu propio tablero aquí
tablero_custom = np.array([
    [0, 0, 0, 0, 0, 0, 0, 0],
    [0, 0, 0, 0, 0, 0, 0, 0],
    [0, 0, 1, 1, 1, 0, 0, 0],
    [0, 0, 1, 1, 1, 0, 0, 0],
    [0, 0, 1, -1, -1, -1, 0, 0],
    [0, 0, 0, -1, -1, 0, 0, 0],
    [0, 0, 0, 0, 0, 0, 0, 0],
    [0, 0, 0, 0, 0, 0, 0, 0]
])

print("TABLERO PERSONALIZADO:")
imprimir_tablero(tablero_custom)

# Inspeccionar features
top_features = inspect_top_features(tablero_custom, sae, top_k=5)

### Ejemplo 3: Comparar Diferentes Configuraciones

In [ ]:
# Tablero con control de esquinas
tablero_esquinas = np.array([
    [1, -1, 0, 0, 0, 0, -1, 1],
    [-1, -1, 0, 0, 0, 0, -1, -1],
    [0, 0, 0, 0, 0, 0, 0, 0],
    [0, 0, 0, 1, -1, 0, 0, 0],
    [0, 0, 0, -1, 1, 0, 0, 0],
    [0, 0, 0, 0, 0, 0, 0, 0],
    [1, 1, 0, 0, 0, 0, 1, 1],
    [-1, 1, 0, 0, 0, 0, 1, -1]
])

print("TABLERO CON ESQUINAS CONTROLADAS:")
imprimir_tablero(tablero_esquinas)

top_features = inspect_top_features(tablero_esquinas, sae, top_k=5)

## 5. Análisis de Features Específicas

Una vez identificadas las features más activas, puedes:
- Comparar con el ground truth de BSPs
- Ver qué tableros activan cada feature
- Calcular métricas de interpretabilidad

In [ ]:
# TODO: Agregar análisis de features específicas
# Por ejemplo, cargar el ground truth y ver qué BSPs corresponden a cada feature